# Swiggy Flat Bot — Fine-tuning Qwen2.5-7B for Intent Parsing

**Before running:** copy `train.jsonl` from `synthetic_data/` into this folder.

```
training/
├── finetune.ipynb   ← this file
├── requirements.txt
├── tools.json
└── train.jsonl      ← copy here
```

**Outputs created automatically:**
```
training/
├── checkpoints/     ← saved per epoch during training
├── output/lora/     ← final LoRA adapter
├── output/merged/   ← full merged model (optional)
└── output/gguf/     ← GGUF files for Ollama
```

## 0. GPU Check
Run this first to confirm your setup.

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'GPU count       : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB')

## 1. Install Dependencies
Only run once. Restart kernel after installation.

In [ ]:
# Run once, then restart kernel
import subprocess
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Done. Restart kernel now if this was the first install.')

## 2. Configuration
All tunable parameters in one place.

In [ ]:
import os

# --- Model ---
MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'  # pre-quantized, downloads faster
# Use 'unsloth/Qwen2.5-7B-Instruct' if you want full precision before QLoRA

MAX_SEQ_LENGTH = 2048   # enough for system prompt + short message + tool call
LOAD_IN_4BIT   = True   # QLoRA — set False if you have 24+ GB VRAM and want full LoRA

# --- LoRA ---
LORA_R         = 16     # rank; 32 gives slightly better quality but uses more memory
LORA_ALPHA     = 16     # keep equal to LORA_R
LORA_DROPOUT   = 0.0

# --- Training ---
NUM_EPOCHS     = 5      # small dataset (480 examples) needs more epochs
BATCH_SIZE     = 2      # per GPU; increase to 4 if VRAM allows
GRAD_ACCUM     = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE  = 2e-4
WARMUP_STEPS   = 10
EVAL_SPLIT     = 0.1    # 10% held out for validation
SEED           = 42

# --- Paths ---
TRAIN_FILE     = 'train.jsonl'
CKPT_DIR       = 'checkpoints'
LORA_DIR       = 'output/lora'
MERGED_DIR     = 'output/merged'
GGUF_DIR       = 'output/gguf'
GGUF_NAME      = 'swiggy-intent'
TOOLS_FILE     = 'tools.json'

# --- GPU selection (optional) ---
# If you have multiple GPUs and want to use only one:
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

for d in [CKPT_DIR, LORA_DIR, MERGED_DIR, GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config ready.')

## 3. Load Model & Tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,           # auto-detect: bf16 on Ampere+, fp16 otherwise
    load_in_4bit    = LOAD_IN_4BIT,
)

print(f'Model loaded: {MODEL_NAME}')
print(f'dtype       : {model.dtype}')
print(f'Device      : {next(model.parameters()).device}')

## 4. Load & Format Dataset

Each example is formatted using the model's native chat template with tool definitions injected,
so the model learns the exact output format it will use at inference time.

In [ ]:
import json
from datasets import Dataset

# Load tool schemas — injected into every training example
with open(TOOLS_FILE) as f:
    TOOLS = json.load(f)

# Load JSONL
raw = []
with open(TRAIN_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            raw.append(json.loads(line))

print(f'Loaded {len(raw)} examples from {TRAIN_FILE}')

# Action distribution
from collections import Counter
actions = [ex['messages'][2]['tool_calls'][0]['function']['name'] for ex in raw]
print('\nAction distribution:')
for action, count in sorted(Counter(actions).items(), key=lambda x: -x[1]):
    print(f'  {action:<35} {count}')

In [ ]:
def format_example(example):
    """
    Apply the model's chat template with tool schemas.
    The template injects tool definitions into the system message automatically.
    """
    text = tokenizer.apply_chat_template(
        example['messages'],
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

# Apply template
dataset = Dataset.from_list(raw)
dataset = dataset.map(format_example, remove_columns=['messages'])

# Train / eval split
split = dataset.train_test_split(test_size=EVAL_SPLIT, seed=SEED)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'\nTrain examples : {len(train_dataset)}')
print(f'Eval examples  : {len(eval_dataset)}')

# Preview one formatted example
print('\n--- Sample formatted text (first 800 chars) ---')
print(train_dataset[0]['text'][:800])
print('...')

## 5. Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # saves VRAM
    random_state   = SEED,
)

# Show trainable parameter count
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}  ({100 * trainable / total:.2f}%)')

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_dataset,
    eval_dataset      = eval_dataset,
    dataset_text_field= 'text',
    max_seq_length    = MAX_SEQ_LENGTH,
    dataset_num_proc  = 2,
    packing           = False,
    args = TrainingArguments(
        output_dir                  = CKPT_DIR,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = SEED,
        logging_steps               = 5,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 2,        # keep only the 2 best checkpoints
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        report_to                   = 'none',   # set 'wandb' if you want W&B logging
    ),
)

print('Starting training...')
trainer_stats = trainer.train()

print(f'\nTraining complete.')
print(f'Runtime       : {trainer_stats.metrics["train_runtime"]:.0f}s')
print(f'Train loss    : {trainer_stats.metrics["train_loss"]:.4f}')

## 7. Save LoRA Adapter
The LoRA adapter is small (~100MB). Save it first before the heavier steps.

In [ ]:
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'LoRA adapter saved to: {LORA_DIR}')

## 8. Export to GGUF (for Ollama)

This merges the LoRA weights into the base model and quantizes to Q4_K_M (~4.5 GB).  
This is the file you'll load into Ollama.

**Quantization options** (trade quality vs. size):
| Method | Size | Use when |
|--------|------|----------|
| `q2_k` | ~2.7 GB | very low RAM |
| `q4_k_m` | ~4.5 GB | **recommended** |
| `q5_k_m` | ~5.1 GB | more VRAM, better quality |
| `q8_0` | ~7.7 GB | highest quality |
| `f16` | ~14 GB | no quantization |

In [ ]:
gguf_path = f'{GGUF_DIR}/{GGUF_NAME}'

print('Exporting to GGUF (Q4_K_M)... this takes a few minutes.')
model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method='q4_k_m',
)
print(f'GGUF saved to: {gguf_path}-Q4_K_M.gguf')

## 9. (Optional) Save Merged Full Model
Only needed if you want the full HuggingFace model (for vLLM, HF Inference, etc.).
Skip this if you're only using Ollama.

In [ ]:
# Uncomment to run
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved to: {MERGED_DIR}')

## 10. Deploy to Ollama

Run the cells below to generate the Modelfile and get the exact commands to run on your machine.

In [ ]:
import os, glob

# Find the generated GGUF file
gguf_files = glob.glob(f'{GGUF_DIR}/*.gguf')
if not gguf_files:
    print('No GGUF file found. Run cell 8 first.')
else:
    gguf_file = gguf_files[0]
    gguf_abs  = os.path.abspath(gguf_file)
    print(f'Found GGUF: {gguf_abs}')

    system_prompt = (
        'You are a grocery ordering assistant for a shared flat Telegram group connected to Swiggy Instamart. '
        'Always call the most specific tool available. '
        'If the message is a standalone greeting with NO grocery intent, always use `respond`. '
        'Use `respond` for: greetings, thanks, compliments, questions unrelated to groceries, or anything ambiguous. '
        'The users often write in Hindi or Hinglish.'
    )

    modelfile_content = f'FROM {gguf_abs}\nSYSTEM "{system_prompt}"\n'
    modelfile_path = f'{GGUF_DIR}/Modelfile'

    with open(modelfile_path, 'w') as f:
        f.write(modelfile_content)

    print(f'\nModelfile written to: {modelfile_path}')
    print('\n--- Run these commands on your machine ---')
    print(f'ollama create swiggy-intent -f {os.path.abspath(modelfile_path)}')
    print(f'ollama run swiggy-intent')
    print('\n--- Then update your .env ---')
    print('OLLAMA_MODEL=swiggy-intent')
    print('USE_GEMINI=false')
    print('USE_GROQ=false')

## 11. Quick Inference Test
Test the fine-tuned model directly in the notebook before deploying.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # switch to fast inference mode

SYSTEM_PROMPT = (
    'You are a grocery ordering assistant for a shared flat Telegram group connected to Swiggy Instamart. '
    'Always call the most specific tool available. '
    'If the message is a standalone greeting with NO grocery intent, always use `respond`. '
    'The users often write in Hindi or Hinglish.'
)

test_messages = [
    'bhai 2 ande aur doodh daal do',
    'roz raat 9 baje order set kar do',
    'hey',
    'cart dikhao, kitna total hua',
    'amul butter kitne ka hai',
    'abhi order kar do',
    'kal subah 8 baje karna order',
]

print('=== Inference Test ===\n')
for msg in test_messages:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'User    : {msg}')
    print(f'Output  : {response.strip()}')
    print()